In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from catboost import CatBoostRegressor

In [1]:
# ==========================================================
# GRIDLOCK HACKATHON FINAL SUBMISSION
# LOOKUP + CATBOOST BLEND
# ==========================================================

!pip install -q catboost

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from catboost import CatBoostRegressor

# ==========================================================
# LOAD DATA
# ==========================================================

TRAIN_PATH = "/kaggle/input/datasets/rohanthakar/traffic-demand-prediction-gridlock/dataset/train.csv"
TEST_PATH = "/kaggle/input/datasets/rohanthakar/traffic-demand-prediction-gridlock/dataset/test.csv"

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

print("Train:", train.shape)
print("Test :", test.shape)

# ==========================================================
# TIME FEATURES
# ==========================================================

for df in [train, test]:

    df["timestamp"] = pd.to_datetime(
        df["timestamp"],
        format="%H:%M"
    )

    df["hour"] = df["timestamp"].dt.hour

    df["minute"] = df["timestamp"].dt.minute

    df["time_slot"] = (
        df["hour"] * 4
        + df["minute"] // 15
    )

    df["hour_sin"] = np.sin(
        2 * np.pi * df["hour"] / 24
    )

    df["hour_cos"] = np.cos(
        2 * np.pi * df["hour"] / 24
    )

# ==========================================================
# MISSING VALUES
# ==========================================================

for col in [
    "RoadType",
    "LargeVehicles",
    "Landmarks",
    "Weather"
]:

    train[col] = (
        train[col]
        .fillna("Unknown")
        .astype(str)
    )

    test[col] = (
        test[col]
        .fillna("Unknown")
        .astype(str)
    )

temp_median = train["Temperature"].median()

train["Temperature"] = (
    train["Temperature"]
    .fillna(temp_median)
)

test["Temperature"] = (
    test["Temperature"]
    .fillna(temp_median)
)

# ==========================================================
# INTERACTION FEATURES
# ==========================================================

for df in [train, test]:

    df["geo_hour"] = (
        df["geohash"]
        + "_"
        + df["hour"].astype(str)
    )

    df["road_weather"] = (
        df["RoadType"]
        + "_"
        + df["Weather"]
    )

    df["lane_temp"] = (
        df["NumberofLanes"]
        * df["Temperature"]
    )

# ==========================================================
# DAY 48 LOOKUP TABLE
# ==========================================================

train48 = train[
    train["day"] == 48
].copy()

lookup = train48.set_index(
    ["geohash", "time_slot"]
)["demand"]

lookup_pred = test.set_index(
    ["geohash", "time_slot"]
).index.map(lookup)

lookup_pred = pd.Series(
    lookup_pred,
    index=test.index
)

print(
    "Lookup Coverage:",
    lookup_pred.notna().mean()
)

# ==========================================================
# HIERARCHICAL FALLBACKS
# ==========================================================

geo_mean = (
    train48.groupby("geohash")
    ["demand"]
    .mean()
)

slot_mean = (
    train48.groupby("time_slot")
    ["demand"]
    .mean()
)

hour_mean = (
    train48.groupby("hour")
    ["demand"]
    .mean()
)

global_mean = (
    train48["demand"]
    .mean()
)

missing = lookup_pred.isna()

lookup_pred.loc[missing] = (
    test.loc[missing, "geohash"]
    .map(geo_mean)
)

missing = lookup_pred.isna()

lookup_pred.loc[missing] = (
    test.loc[missing, "time_slot"]
    .map(slot_mean)
)

missing = lookup_pred.isna()

lookup_pred.loc[missing] = (
    test.loc[missing, "hour"]
    .map(hour_mean)
)

lookup_pred = lookup_pred.fillna(
    global_mean
)

# ==========================================================
# CATBOOST FEATURES
# ==========================================================

FEATURES = [
    "geohash",
    "RoadType",
    "NumberofLanes",
    "LargeVehicles",
    "Landmarks",
    "Temperature",
    "Weather",
    "day",
    "hour",
    "minute",
    "time_slot",
    "hour_sin",
    "hour_cos",
    "lane_temp",
    "geo_hour",
    "road_weather"
]

cat_features = [
    FEATURES.index("geohash"),
    FEATURES.index("RoadType"),
    FEATURES.index("LargeVehicles"),
    FEATURES.index("Landmarks"),
    FEATURES.index("Weather"),
    FEATURES.index("geo_hour"),
    FEATURES.index("road_weather")
]

# ==========================================================
# TRAIN ON ALL DATA
# ==========================================================

model = CatBoostRegressor(
    iterations=5000,
    learning_rate=0.02,
    depth=10,
    loss_function="RMSE",
    random_seed=42,
    verbose=500
)

model.fit(
    train[FEATURES],
    train["demand"],
    cat_features=cat_features
)

# ==========================================================
# CATBOOST PREDICTIONS
# ==========================================================

cat_pred = model.predict(
    test[FEATURES]
)

cat_pred = np.clip(
    cat_pred,
    0,
    1
)

# ==========================================================
# FINAL BLEND
# ==========================================================

final_pred = (
    0.65 * lookup_pred.values
    + 0.35 * cat_pred
)

final_pred = np.clip(
    final_pred,
    0,
    1
)

# ==========================================================
# SUBMISSION
# ==========================================================

submission = pd.DataFrame({
    "Index": test["Index"],
    "demand": final_pred
})

submission.to_csv(
    "submission.csv",
    index=False
)

print("\nSubmission Shape:")
print(submission.shape)

print("\nMissing Values:")
print(submission.isnull().sum())

print("\nPrediction Range:")
print(
    submission["demand"].min(),
    submission["demand"].max()
)

print("\nSaved: submission.csv")

submission.head()

Train: (77299, 11)
Test : (41778, 10)
Lookup Coverage: 0.8888888888888888
0:	learn: 0.1397703	total: 194ms	remaining: 16m 9s
500:	learn: 0.0350921	total: 48.5s	remaining: 7m 15s
1000:	learn: 0.0311859	total: 1m 39s	remaining: 6m 38s
1500:	learn: 0.0289695	total: 2m 31s	remaining: 5m 53s
2000:	learn: 0.0272830	total: 3m 23s	remaining: 5m 4s
2500:	learn: 0.0260333	total: 4m 16s	remaining: 4m 15s
3000:	learn: 0.0249544	total: 5m 8s	remaining: 3m 25s
3500:	learn: 0.0240091	total: 6m 1s	remaining: 2m 34s
4000:	learn: 0.0231928	total: 6m 55s	remaining: 1m 43s
4500:	learn: 0.0224234	total: 7m 49s	remaining: 52s
4999:	learn: 0.0217429	total: 8m 42s	remaining: 0us

Submission Shape:
(41778, 2)

Missing Values:
Index     0
demand    0
dtype: int64

Prediction Range:
0.004025063083262655 1.0

Saved: submission.csv


,Index,demand
0,0,0.043539
1,1,0.029642
2,2,0.021129
3,3,0.055024
4,4,0.087077


In [2]:
submission["demand"].describe()

count    41778.000000
mean         0.112156
std          0.152328
min          0.004025
25%          0.025610
50%          0.060864
75%          0.130803
max          1.000000
Name: demand, dtype: float64